<a href="https://colab.research.google.com/github/sajibmdsaberahmad-create/trading-bot-Grandmaster/blob/main/colab_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 HA-NUN Grandmaster Training on Google Colab

This notebook trains the **210M Teacher → 21M Student** distillation pipeline on Google Colab.

## Instructions
1. Open in Google Colab
2. Set Runtime → Change runtime type → Hardware accelerator: **GPU**
3. Run cells **1-6** sequentially (Shift + Enter)
4. Optional: run cells 7-9 for multi-timeframe and LSE training

**Note**: If you see `fatal: could not read Username` or clone failures, use the fix in COLAB_FIX.md or ensure the repo is public.

In [1]:
# STEP 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify mount
!ls -la /content/drive/MyDrive/ | head -20

Mounted at /content/drive
total 10761
-rw------- 1 root root 1190519 Sep 22  2022 20210910_083639.jpg
-rw------- 1 root root     176 Jun 19  2025 can you analize the codes deeply.gsheet
-rw------- 1 root root     176 Jun 25  2025 can you give me full files of your updated codes... (1).gdoc
-rw------- 1 root root     176 Jun 25  2025 can you give me full files of your updated codes....gdoc
-rw------- 1 root root   66446 May  9  2024 cc00e803-baa9-49a1-8212-072baec32def.jpeg
drwx------ 2 root root    4096 Jun 22 07:27 Colab Notebooks
drwx------ 2 root root    4096 Nov  4  2025 EBPU Books
-rw------- 1 root root  189690 Sep 22  2022 EC Bangladesh_1625640658296.jpg
-rw------- 1 root root  189690 Sep 22  2022 EC Bangladesh_1625640668825.jpg
-rw------- 1 root root   77717 Sep 22  2022 FB_IMG_1612693071396.jpg
-rw------- 1 root root   77717 Sep 22  2022 FB_IMG_1612693148383.jpg
-rw------- 1 root root   76984 Sep 22  2022 FB_IMG_1616088764841.jpg
-rw------- 1 root root   76984 Sep 22  2022 FB_I

In [ ]:
# STEP 2: Navigate and clone repository
%cd /content/drive/MyDrive/

GITHUB_USERNAME = "sajibmdsaberahmad-create"
REPO_NAME = "trading-bot-HA-NUN"
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

import os
if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL}
else:
    print("Repo already exists, pulling latest...")
    %cd {REPO_NAME}
    !git pull origin main
    %cd ..

# Enter repo
%cd {REPO_NAME}
!pwd
!ls -la

# Verify repo contents
!test -f main.py && echo "✅ Repo ready" || echo "❌ main.py not found in repo root"

/content/drive/MyDrive
Repo already exists, pulling latest...
/content/drive/MyDrive/trading-bot-HA-NUN
remote: Enumerating objects: 112, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 98 (delta 67), reused 76 (delta 45), pack-reused 0 (from 0)
Unpacking objects: 100% (98/98), 25.76 KiB | 17.00 KiB/s, done.
From https://github.com/sajibmdsaberahmad-create/trading-bot-HA-NUN
 * branch            main       -> FETCH_HEAD
   7e98eaf..011bd28  main       -> origin/main
Updating 7e98eaf..011bd28


In [ ]:
# STEP 3: Install system dependencies
!pip install -r requirements.txt

# Verify GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be slow.")

In [ ]:
# STEP 4: Run Grandmaster Distillation Training (210M → 21M)

TICKER = "SPY"           # Ticker to train on
PPO_TIMESTEPS = 1000000   # 1M timesteps for deep learning
EPOCHS = 50               # Transformer/LSTM epochs
DEVICE = "auto"           # auto-detect GPU

print("=" * 70)
print("  🧠 GRANDMASTER DISTILLATION TRAINING")
print("=" * 70)
print(f"Ticker: {TICKER}")
print(f"PPO Timesteps: {PPO_TIMESTEPS:,}")
print(f"Epochs: {EPOCHS}")
print(f"Device: {DEVICE}")
print("Working dir: ")
!pwd
print("=" * 70)

# Run the training pipeline
!python main.py --mode advanced-train \
    --ticker {TICKER} \
    --ppo-timesteps {PPO_TIMESTEPS} \
    --epochs {EPOCHS} \
    --device {DEVICE} \
    --train-start 2020-01-01 \
    --train-end 2024-12-01 \
    --use-synthetic

In [ ]:
# STEP 5: Verify trained models exist
!ls -lah models/
!ls -lah models/checkpoints/
!ls -lah models/backups/

# Show training history
import json
try:
    with open('models/training_history.json', 'r') as f:
        history = json.load(f)
    print("\nTraining History:")
    print(f"Models trained: {history.get('models_trained', [])}")
    print(f"Data: {history.get('data', {})}")
    print(f"Metrics: {history.get('metrics', {})}")
except FileNotFoundError:
    print("No training history found yet.")

In [ ]:
# STEP 6: Commit and push trained weights to GitHub

!git config --global user.email "colab@ha-nun.ai"
!git config --global user.name "HA-NUN Colab Training"

# Stage all model files (ignore missing)
!git add models/*.pth models/*.h5 models/*.json models/checkpoints/* models/backups/* models/training_history.json || true

# Check status
!git status --short

# Commit
!git commit -m "train: grandmaster distillation weights updated via Colab (210M->21M)" || echo "Nothing to commit"

# Push
!git push origin main || echo "Push failed. Check auth or run manually."

In [ ]:
# OPTIONAL: Multi-Timeframe Training
TIMEFRAMES = [
    {"name": "1min_scalper", "ticker": "SPY", "timeframe": "1min", "ppo_timesteps": 500000},
    {"name": "5min_scalper", "ticker": "SPY", "timeframe": "5min", "ppo_timesteps": 500000},
    {"name": "1h_swing", "ticker": "SPY", "timeframe": "1h", "ppo_timesteps": 300000},
    {"name": "4h_swing", "ticker": "SPY", "timeframe": "4h", "ppo_timesteps": 300000},
    {"name": "1d_position", "ticker": "SPY", "timeframe": "1d", "ppo_timesteps": 200000},
]

for tf_config in TIMEFRAMES:
    print(f"\n{'='*70}")
    print(f"  Training {tf_config['name']}...")
    print(f"{'='*70}")
    !python main.py --mode advanced-train \
        --ticker {tf_config['ticker']} \
        --timeframe {tf_config['timeframe']} \
        --ppo-timesteps {tf_config['ppo_timesteps']} \
        --epochs 30 \
        --device auto \
        --use-synthetic
    !cp models/transformer_model.pth models/transformer_model_{tf_config['name']}.pth
    !cp models/lstm_model.h5 models/lstm_model_{tf_config['name']}.h5
    !cp ppo_trader.zip ppo_trader_{tf_config['name']}.zip

In [ ]:
# OPTIONAL: LSE Training (London Stock Exchange)
LSE_TICKERS = [
    "ISF",   # iShares Core FTSE 100 UCITS ETF
    # "VOD",  # Vodafone
    # "BP",   # BP
]

for lse_ticker in LSE_TICKERS:
    print(f"\n{'='*70}")
    print(f"  Training LSE: {lse_ticker}")
    print(f"{'='*70}")
    !python main.py --mode advanced-train \
        --ticker {lse_ticker} \
        --lse \
        --timeframe 1h \
        --ppo-timesteps 300000 \
        --epochs 30 \
        --device auto \
        --use-synthetic
    !cp models/transformer_model.pth models/transformer_model_LSE_{lse_ticker}.pth
    !cp models/lstm_model.h5 models/lstm_model_LSE_{lse_ticker}.h5
    !cp ppo_trader.zip ppo_trader_LSE_{lse_ticker}.zip
    print(f"✅ LSE {lse_ticker} training complete")

In [ ]:
# FINAL: Push all new models to GitHub
!git add models/*.pth models/*.h5 models/*.zip models/*.json || true
!git add backtest_results/*.json || true
!git status --short
!git commit -m "feat: multi-timeframe + LSE grandmaster training complete" || echo "Nothing to commit"
!git push origin main || echo "Push manually if needed"